# Anvendt KI og ML: Recurrent Neural Networks (RNN)
Welcome to this notebook which is part of Kristiania University College's course «Anvendt kunstig intelligens og maskinlæring», module 5, «Dyp læring og overføringslæring».

## How to use this notebook
This is a Jupyter notebook with text and Python code that you run by pressing shift-enter. The purpose is to become familiar with relevant concepts in practice, not necessarily to understand all the details of the code. Take the time to read the text, review the code, run the code cells (make sure they are in the correct order), study the results, and think about what we do, why and how.

In this notebook, you will find self-test questions in _italics_. Go to Canvas and answer the questions there before continuing with the notebook.

## The problem: Movie review sentiment analysis
The task is to perform sentiment analysis on movie reviews, determining if they are positive or negative.

## Dataset information
We will use a famous dataset for sentiment analysis, called the IMDb Large Movie Review Dataset. It is a dataset for binary sentiment classification containing 25,000 highly polarized movie reviews for training, and 25,000 for testing. 

![IMDb](figures/imdb.png)

## 1. Importing Libraries

In [ ]:
#Importing libraries
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt
from tensorflow.keras.layers import LSTM

## 2. Loading the dataset from Keras

In [ ]:
# Note: There is a bug on some systems that may affect this notebook.
# If you get a «Kernel Restarting» error message when running this notebook,
# uncomment the code below (i.e. remove the hash signs in the beginning of the
# code lines). Then restart the kernel and rerun the entire notebook from here.

#import os
#os.environ['KMP_DUPLICATE_LIB_OK']='True'

The IMDb dataset has been preprocessed. Each review has the format of an **indexed bag of words**, which means that, instead of plain text, it is stored as a list of IDs of its most frequent words. The ID-word correspondence are stored in a separate dictionary. The most frequent words come first and rare words are ignored.

Let's take a look at this format.

In [ ]:
#Loading indexed reviews
vocabulary_size = 5000

(input_train, y_train), (input_test, y_test) = imdb.load_data(num_words = vocabulary_size)
print('Loaded dataset with {} training samples, {} test samples'.format(len(input_train), len(input_test)))

In [ ]:
#Loading the word:id dictionary and reversing it to id:word
word2id = imdb.get_word_index()
id2word = {i: word for word, i in word2id.items()}

In [ ]:
#Example of an input sample (bag of words) and its label
decoded_bag = [id2word.get(i, ' ') for i in input_train[0]]
print('---review with words---')
print(decoded_bag)

print('\n---label---')
print(y_train[6])

We have mapped the review to the original words and also revealed its label. We see that the label is 1, which means that it is a positive review.

_Oppgave: Svar på fritekstspørsmål 1 (vist nedenfor) i Canvas._

1. _What do you think are the advantages of using an indexed bag of words? Can you think of any disadvantages?_

## 3. Adding a pad sequence
In order to feed this data into our RNN, all input documents must have the same length. We can see the range of the length reviews:

In [ ]:
all_reviews = np.concatenate([input_train, input_test])

In [ ]:
lengths = [len(i) for i in all_reviews]
print('Minimum review length: {}'.format(min(lengths)))
print('Maximum review length: {}'.format(max(lengths)))

_Oppgave: Svar på fritekstspørsmål 2 (vist nedenfor) i Canvas._

2. _Using what you have learned in previous exercises, can you plot the distribution of the input lengths? How would you describe the distribution?_

We will limit the maximum review length to `max_words` by truncating longer reviews and padding shorter reviews with a null value (0).

In [ ]:
from tensorflow.keras.preprocessing import sequence
max_words = 500
input_train = sequence.pad_sequences(input_train, maxlen=max_words)
input_test = sequence.pad_sequences(input_test, maxlen=max_words)
print('input_train shape:', input_train.shape)
print('input_test shape:', input_test.shape)

In [ ]:
print('---review with words---')
print([id2word.get(i, ' ') for i in input_train[6]])
print('---label---')
print(y_train[6])

We can check the lengths again to see that they all have the same lengths.

In [ ]:
all_reviews = np.concatenate([input_train, input_test])
lengths = [len(i) for i in all_reviews]
print('Minimum review length: {}'.format(min(lengths)))
print('Maximum review length: {}'.format(max(lengths)))

_Oppgave: Svar på fritekstspørsmålene 3 og 4 (vist nedenfor) i Canvas._

3. _What are the disadvantages of adjusting the samples’ lengths?_
4. _What are the advantages of such an approach? (Tip: Think about the bias-variance trade-off.)_

## 4. Defining the model using Keras

Next, we define the layers of the neural network. One of the layers we will use, is a Keras `SimpleRNN` layer.

In [ ]:
max_features = 10000
model = Sequential()
model.add(Embedding(max_features, 32))
model.add(SimpleRNN(32))
model.add(Dense(1, activation='sigmoid'))

_Oppgave: Svar på fritekstspørsmål 5 (vist nedenfor) i Canvas._

5. _Can you explain why we used a sigmoid function as the activation function in the output layer?_

Let us display the architecture of the network so far:

In [ ]:
model.summary()

## 5. Compiling and fitting the model on the training data
We will now train the Recurrent Neural Network on the IMDB review data.

During the next step of compiling the model, we specify the following:
1. `optimizer='rmsprop'` – specifies that we want to use rmsprop optimization during training. 
2. `loss=categorical_crossentropy` – the loss function we want to use in this case is the [cross entropy loss](https://www.tensorflow.org/api_docs/python/tf/keras/losses/SparseCategoricalCrossentropy) used in categorical classification. 
3. `metrics=['acc']` – sets which metrics to track during training.

In [ ]:
#Compiling Model
model.compile(optimizer='rmsprop',
              loss='binary_crossentropy', metrics=['acc'])

Now, let's train the model by calling `.fit()`. All we need to pass is train_images and associated labels, the number of iterations (epochs) and the batch size. These are defined as:

- **Epochs**: The number of times the weights in the network will be updated. The value should be large enough for the network to converge (minimize loss to a stable level). If you run the training again without redefining the network, the training will continue from the previous session.

- **Batch size**: Determines the number of samples for cross-validation. Generally you should use as small a batch size as patience/available computing time allows for.

In [ ]:
#Fitting Model
history = model.fit(input_train, y_train,
                    epochs=10,
                    batch_size=128,
                    validation_split=0.2)

## 6. Evaluating the model performance on the validation data

In [ ]:
acc = history.history['acc']
val_acc = history.history['val_acc']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc) + 1)

print(f'Avearage Validation Loss:{np.round(np.mean(val_loss), 2)}\n Average Validation Accuracy:{np.round(np.mean(val_acc),2)}')

In [ ]:
# plotting the training and validation accuracy
plt.plot(epochs, acc, 'ro', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.legend()
plt.figure()
plt.plot(epochs, loss, 'ro', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()
plt.show()

> Did you get a «Kernel Restarting» error message here? If so, go to the first code cell of this notebook, uncomment the code, restart the kernel, and try again. If that doesn't work, contact your supervisor.

This small recurrent neural network achieves the average validation accuracy of around 80 %. Let’s look at some more advanced layers and see if they can perform better.

## Long Short Term Memory

## 1. Defining the model using Keras

In [ ]:
# Define Model
model = Sequential()
model.add(Embedding(max_features, 32))
model.add(LSTM(32))
model.add(Dense(1, activation='sigmoid'))

Let us display the architecture of the network so far:

In [ ]:
model.summary()

## 2. Compiling and fitting the model on the training data
We will now train the LSTM on the IMDB review data.

In [ ]:
# Compile Model
model.compile(optimizer='rmsprop',
              loss='binary_crossentropy',
              metrics=['acc'])

Now, let's train the model by calling `.fit()`. All we need to pass is train_images and associated labels, the number of iterations (epochs) and the batch size.

In [ ]:
# Fit Model
history = model.fit(input_train, y_train,
                    epochs=10,
                    batch_size=128,
                    validation_split=0.2)

## 3. Evaluating the model performance on the validation data

In [ ]:
acc = history.history['acc']
val_acc = history.history['val_acc']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc) + 1)

print(f'Average Validation Loss:{np.round(np.mean(val_loss), 2)}\nAverage Validation Accuracy:{np.round(np.mean(val_acc),2)}')

In [ ]:
plt.plot(epochs, acc, 'ro', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.legend()
plt.figure()
plt.plot(epochs, loss, 'ro', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()
plt.show()

This time, we achieved up to 89 % validation accuracy. Not bad: certainly much better than the SimpleRNN network. We can further increase the accuracy by tuning hyperparameters such as dimensionality of embeddings or the LSTM output.

_Oppgave: Svar på fritekstspørsmål 6 (vist nedenfor) i Canvas._

6. _Can you think of a possible explanation of why LSTM works better than SimpleRNN for this data?_